In [ ]:
#===============================================================================
#
# Name: birdapp-amazon-ntlk.ipynb
#
# Author: MT
#
# Date: 23/11/2025
#
# Purpose: Code Base for NLTK (Natural Language Toolkit)
#
# Links:
#       https://www.datacamp.com/tutorial/text-analytics-beginners-nltk
#       https://towardsdatascience.com/text-summarization-using-tf-idf-e64a0644ace3
#       https://github.com/cjhutto/vaderSentiment
#       https://raw.githubusercontent.com/pycaret/pycaret/master/datasets/amazon.csv
#
# Steps:
# Step 0: Imports
# Step 1: Read Data
# Step 2: Preprocess Data
# Step 3: Sentiment Analysis;
# Step 4: Word Cloud;
# Step 5: Review Importance - Word Frequency;
# Step 6: Term-Frequency Inverse-Document Frequency (TF-IDF) - Text Summarization;
# Step 7a: Latent Dirichlet Allocation (LDA)
# Step 7b: Latent Dirichlet Allocation (LDA) - 2 topics
#
#===============================================================================

In [2]:
#===============================================================================
# Step 0: Imports;

#--------------------
# Pandas, Numpy, copy and os
import pandas as pd
import numpy as np
import copy
import os
pd.options.display.float_format = '{:.4f}'.format # suppress scientific notation
#--------------------

#--------------------
# Plotly Express
import plotly.express as px
#--------------------

#--------------------
# NLTK
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk import ngrams
from nltk.probability import FreqDist
nltk.data.path.append('C:/temp/nltk_data')
#--------------------

#--------------------
# String manipulations
import string
import re
import contractions
#--------------------

#--------------------
# WordCloud
from wordcloud import WordCloud
import matplotlib.pyplot as plt
#--------------------

#--------------------
# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
#--------------------

#--------------------
import gensim
from gensim import corpora
from gensim.models import CoherenceModel
#--------------------
#===============================================================================

ModuleNotFoundError: No module named 'contractions'

In [ ]:
#===============================================================================
# Step 1: Read Data;
df = pd.read_csv('amazon.csv')
print(df.info())
display(df)
#===============================================================================

In [ ]:
#===============================================================================
# Step 2: Preprocess Data;

#--------------------
# Copy the data
df2=df.copy()
#--------------------

#--------------------
print('English Stopwords:')
print(stopwords.words('english'))
print('')
print('English Punctuation:')
print(string.punctuation)
print('')
#--------------------

#--------------------
# Lower Case
df2['reviewText_lower']=df2.reviewText.str.lower()
#--------------------

#--------------------
# Expand contractions
df2['reviewText_lower_contractions']=df2['reviewText_lower'].apply(lambda x: contractions.fix(x))
#--------------------

#--------------------
# Remove stopwords

# Get the list of stopwords
stop_words = stopwords.words('english') + ["&#34;"]

# Tokenize the text in each row
df2['prep'] = df2['reviewText_lower_contractions'].apply(word_tokenize)

# Remove stopwords
df2['prep2'] = df2['prep'].apply(lambda tokens: [token for token in tokens if token.lower() not in stop_words])

# Join the tokens back into strings if needed
df2['reviewText_lower_contractions_tokenized'] = df2['prep2'].apply(' '.join)
#--------------------

#--------------------
# Remove punctuation using string.punctuation for simple patterns
df2['reviewText_lower_contractions_tokenized_punct'] = df2['reviewText_lower_contractions_tokenized'].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))
#--------------------

#--------------------
# Lemmatize text
lemmatizer = WordNetLemmatizer()
df2['reviewText_lower_contractions_tokenized_punct_lemmatized'] = df2['reviewText_lower_contractions_tokenized_punct'].apply(lambda x: ' '.join([lemmatizer.lemmatize(word) for word in word_tokenize(x)]))
#--------------------

#--------------------
# Check some records
print('Positive review (original, contractions, lowercased, stopworded, punctuated):')
print(df2.iloc[0,0])
print(df2.iloc[0,2])
print(df2.iloc[0,3])
print(df2.iloc[0,6])
print(df2.iloc[0,7])
print(df2.iloc[0,8])
print('')

print('Positive review (original, contractions, lowercased, stopworded, punctuated):')
print(df2.iloc[1,0])
print(df2.iloc[1,2])
print(df2.iloc[1,3])
print(df2.iloc[1,6])
print(df2.iloc[1,7])
print(df2.iloc[1,8])
print('')

print('Positive review (original, contractions, lowercased, stopworded, punctuated):')
print(df2.iloc[5,0])
print(df2.iloc[5,2])
print(df2.iloc[5,3])
print(df2.iloc[5,6])
print(df2.iloc[5,7])
print(df2.iloc[5,8])
print('')

print('Negative review (original, contractions, lowercased, stopworded, punctuated):')
print(df2.iloc[96,0])
print(df2.iloc[96,2])
print(df2.iloc[96,3])
print(df2.iloc[96,6])
print(df2.iloc[96,7])
print(df2.iloc[96,8])
print('')
#--------------------
#===============================================================================

In [ ]:
#===============================================================================
# Step 3: Sentiment Analysis;

#--------------------
# Apply VADER sentiment analysis
sia = SentimentIntensityAnalyzer()
df2['Sentiment']=df2['reviewText_lower_contractions_tokenized_punct_lemmatized'].apply(lambda x: sia.polarity_scores(x))
df2[['neg', 'neu', 'pos', 'compound']]=pd.DataFrame(df2['Sentiment'].tolist(), index=df2.index)
#--------------------

#--------------------
# Print 3 most positive reviews
print('Print 3 most positive reviews:')
largest=df2['compound'].nlargest(10).index
for i in largest:
    print(df2.loc[i,'compound'], ': ', df2.loc[i,'reviewText'])
print('')
#--------------------

#--------------------

# Print 3 most negative reviews
print('Print 3 most negative reviews:')
smallest=df2['compound'].nsmallest(10).index
for i in smallest:
    print(df2.loc[i,'compound'], ': ', df2.loc[i,'reviewText'])
print('')
#--------------------
#===============================================================================


In [ ]:
#===============================================================================
# Step 4: Word Cloud;

#--------------------
# Apply the function to generate n-grams using a lambda function
ngram_order=1
df2['unigrams'] = df2['reviewText_lower_contractions_tokenized_punct_lemmatized'].apply(lambda x: ['_'.join(gram) for gram in ngrams(word_tokenize(x), ngram_order)])

ngram_order=2
df2['ngrams'] = df2['reviewText_lower_contractions_tokenized_punct_lemmatized'].apply(lambda x: ['_'.join(gram) for gram in ngrams(word_tokenize(x), ngram_order)])
#--------------------

#--------------------
# Flatten the list of n-grams and create a frequency distribution
all_ngrams = [ngram for sublist in df2['ngrams'] for ngram in sublist]
fdist = FreqDist(all_ngrams)
#--------------------

#--------------------
# Generate the word cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(fdist)
#--------------------

#--------------------
# Plot the word cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Bigram Word Cloud', fontsize=30)
plt.show()
#--------------------

#--------------------
# Most common n-grams
most_common_ngrams = fdist.most_common(30)
most_common_ngrams
#--------------------
#===============================================================================

In [ ]:
#===============================================================================
# Step 5: Review Importance - Word Frequency;

# Calculate an importance score for each review based on the frequency of words in the review divided by the number of words in the review
# https://www.educative.io/answers/text-summarization-in-spacy-and-nltk

#--------------------
# Create unigrams
df2['unigrams'] = df2['reviewText_lower_contractions_tokenized_punct_lemmatized'].apply(lambda x: ['_'.join(gram) for gram in ngrams(word_tokenize(x), 1)])

# Flatten the list of words and create a frequency distribution
all_unigrams = [ngram for sublist in df2['unigrams'] for ngram in sublist]
fdist = FreqDist(all_unigrams)

# fdist as a sorted DataFrame
fdist_df = pd.DataFrame(fdist.items(), columns=['Unigram', 'Frequency']).sort_values(by='Frequency', ascending=False)

# Calculate sentence scores based on word frequencies
def calculate_sentence_scores(sentences, word_frequencies):
    sentence_scores = []
    for sentence in sentences:
        score = 0
        words = word_tokenize(sentence)
        for word in words:
            if word in word_frequencies:
                score += word_frequencies[word]
        sentence_scores.append(score)
    return sentence_scores

# Score each line
df2['scores']=df2['reviewText_lower_contractions_tokenized_punct_lemmatized'].apply(lambda x: calculate_sentence_scores([x], fdist)[0])
#--------------------

#--------------------
# Calculate number of words in each review
df2['Number Of Words']=df2['unigrams'].apply(len)
#--------------------

#--------------------
# Normalize the scores
df2['scoresNorm'] = df2['scores'] / df2['Number Of Words']
#--------------------

#--------------------
# Print 3 most important reviews
print('Print 3 most important reviews:')
largest=df2['scoresNorm'].nlargest(3).index
for i in largest:
    print(df2.loc[i,'scoresNorm'], ': ', df2.loc[i,'reviewText'])
print('')
#--------------------

#--------------------
# Print 3 least important reviews
print('Print 3 least important reviews:')
smallest=df2['scoresNorm'].nsmallest(3).index
for i in smallest:
    print(df2.loc[i,'scoresNorm'], ': ', df2.loc[i,'reviewText'])
print('')
#--------------------
#===============================================================================

In [ ]:
#===============================================================================
# Step 6: Term-Frequency Inverse-Document Frequency (TF-IDF) - Text Summarization;

# https://towardsdatascience.com/text-summarization-using-tf-idf-e64a0644ace3

#--------------------
# Assuming df2 is a DataFrame and 'reviewText_lower_contractions_tokenized_punct_lemmatized' is a column in it
texts = df2['reviewText_lower_contractions_tokenized_punct_lemmatized']

# Initialize the TF-IDF Vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the texts
tfidf_matrix = vectorizer.fit_transform(texts)

# Convert the TF-IDF matrix to a DataFrame for better readability (optional)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# Sum the TF-IDF scores for each document
df2['tfidf_score'] = tfidf_df.sum(axis=1)
#--------------------

#--------------------
# Print 3 most important reviews
print('Print 3 most important reviews:')
largest=df2['tfidf_score'].nlargest(10).index
for i in largest:
    print(df2.loc[i,'tfidf_score'], ': ', df2.loc[i,'reviewText'])
print('')
#--------------------

#--------------------
# Print 3 least important reviews
print('Print 3 least important reviews:')
smallest=df2['tfidf_score'].nsmallest(10).index
for i in smallest:
    print(df2.loc[i,'tfidf_score'], ': ', df2.loc[i,'reviewText'])
print('')
#--------------------
#===============================================================================

In [ ]:
#===============================================================================
# Step 7a: Latent Dirichlet Allocation (LDA)

# Create a dictionary and corpus
dictionary = corpora.Dictionary(df2['unigrams'])
corpus = [dictionary.doc2bow(text) for text in df2['unigrams']]

# Define the range for the number of topics
start = 2
limit = 6
step = 1

# Lists to store coherence values and models
coherence_values = []
model_list = []

# Loop through different numbers of topics
for num_topics in range(start, limit, step):
    # Train the LDA model
    model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                            id2word=dictionary,
                                            num_topics=num_topics,
                                            random_state=100,
                                            update_every=1,
                                            chunksize=100,
                                            passes=10,
                                            alpha='auto',
                                            per_word_topics=True)
    model_list.append(model)
    
    # Compute the coherence score
    coherencemodel = CoherenceModel(model=model, texts=df2['unigrams'], dictionary=dictionary, coherence='c_v')
    coherence_values.append(coherencemodel.get_coherence())

# Graph coherence scores
coherence_df=pd.DataFrame({'Number of Topics': range(start, limit, step), 'Coherence Score':coherence_values})

fig = px.line(
    coherence_df, 
    x='Number of Topics', 
    y='Coherence Score', 
    width=600, height=500,
    title='Coherence Score vs. Number of Topics')
fig.update_traces(mode='lines+markers')
fig.update_xaxes(tickmode='linear', dtick=1)
fig.update_layout(title_x=0.5, title_y=0.85)
fig.show(renderer='notebook')
#===============================================================================

In [ ]:
#===============================================================================
# Step 7b: Latent Dirichlet Allocation (LDA) - 2 topics

#--------------------
# Fit the LDA model
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                        id2word=dictionary,
                                        num_topics=2,
                                        random_state=100,
                                        update_every=1,
                                        chunksize=100,
                                        passes=10,
                                        alpha='auto',
                                        per_word_topics=True)
#--------------------

#--------------------
# Print the topics
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic: {idx} \nWords: {topic}")
#--------------------

#--------------------
# Topic distribution for all documents
all_doc_topics = [lda_model.get_document_topics(doc) for doc in corpus]
all_doc_topics_df = pd.DataFrame([{topic: weight for topic, weight in doc} for doc in all_doc_topics]).fillna(0)
all_doc_topics_df['Topic']=np.where(all_doc_topics_df[1]>0.5,1,0)
display(all_doc_topics_df)
#--------------------

#--------------------
# Create a DataFrame from the topics
topics = lda_model.print_topics(-1, num_words=5)
topics_df = pd.DataFrame(topics, columns=['Topic', 'Words'])

# Function to parse the words and probabilities
def parse_words(words_str):
    words = words_str.split(' + ')
    parsed_words = {}
    for word in words:
        prob, term = word.split('*')
        term = term.strip('"')
        parsed_words[term] = float(prob)
    return parsed_words

# Apply the parsing function to the 'Words' column
parsed_topics = topics_df['Words'].apply(parse_words)

# Stack DataFrame
all_topics_df = pd.DataFrame()
for i in range(len(parsed_topics)):
    topic_df = pd.DataFrame(parsed_topics[i], index=[0]).T.reset_index().rename(columns={'index': 'Word0', 0: 'Probability'})
    topic_df['Topic'] = str(i)
    all_topics_df = pd.concat([all_topics_df, topic_df], ignore_index=True)

all_topics_df['Word'] = all_topics_df['Word0'] + '_' + all_topics_df['Topic']
all_topics_df.sort_values(by=['Topic', 'Probability'], ascending=True, inplace=True)

# Create a vertical bar chart with grouped bars
fig = px.bar(
    all_topics_df, 
    y='Word', 
    x='Probability', 
    color='Topic', 
    title='Topic Words and Probabilities',
    width=600, height=500,
    orientation='h')
fig.update_layout(title_x=0.5, title_y=0.85)
fig.show(renderer='notebook')
#--------------------
#===============================================================================